In [ ]:
import pandas as pd
import random
from datasets import load_dataset
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

selected_subs = [
    'technology', 'programming', 'askscience', 'science', 'philosophy',
    'todayilearned', 'personalfinance', 'buildapc', 'DIY', 'Showerthoughts',
    'tifu', 'explainlikeimfive', 'YouShouldKnow', 'IWantToLearn',
    'LifeProTips', 'gadgets', 'space', 'history', 'books', 'GetMotivated'
]

samples_per_sub = 15000  # Target number of CLEAN posts per subreddit
data_frames = []

for subreddit in selected_subs:
    print(f"\n=== Processing r/{subreddit} (target: {samples_per_sub} clean posts) ===")

    # Load the subreddit split as a streaming dataset
    stream = load_dataset(
        "HuggingFaceGECLM/REDDIT_submissions",
        split=subreddit,
        streaming=True,
        token=HF_TOKEN
    )

    # Collect only clean posts (with selftext that doesn't contain [removed]/[deleted])
    clean_posts = []
    total_processed = 0

    for post in stream:
        total_processed += 1

        # Get selftext, default to empty string if None/missing
        selftext = post.get('selftext', '')

        # Robust filter:
        # 1. Check if selftext exists and is a string
        # 2. Check if it's not empty after stripping whitespace
        # 3. Check that it doesn't contain [removed] or [deleted] (case-insensitive)
        if (selftext and isinstance(selftext, str) and selftext.strip()):
            selftext_lower = selftext.strip().lower()
            if '[removed]' not in selftext_lower and '[deleted]' not in selftext_lower:
                clean_posts.append(post)

        # Stop once we have enough clean posts
        if len(clean_posts) >= samples_per_sub:
            print(f"  Reached target of {samples_per_sub} clean posts")
            print(f"  Processed {total_processed:,} total posts to get {len(clean_posts):,} clean ones")
            break

        # Optional: print progress every 100k processed posts
        if total_processed % 100000 == 0:
            print(f"  Processed {total_processed:,} posts, found {len(clean_posts):,} clean so far")

    # If we couldn't reach the target
    if len(clean_posts) < samples_per_sub:
        print(f"  WARNING: Only found {len(clean_posts)} clean posts (requested {samples_per_sub})")

    print(f"  Collection rate: {len(clean_posts)/total_processed*100:.2f}% of processed posts were clean")

    # Convert to DataFrame and add subreddit column
    df_sub = pd.DataFrame(clean_posts)
    df_sub['subreddit'] = subreddit
    data_frames.append(df_sub)

    print(f"  Added {len(df_sub)} clean posts from r/{subreddit}")

# Combine all subreddits into one DataFrame
combined_df = pd.concat(data_frames, ignore_index=True)
print(f"\n{'='*50}")
print(f"Final combined shape: {combined_df.shape}")
print(f"Total clean posts collected: {len(combined_df)}")

# Quick summary by subreddit
summary = combined_df.groupby('subreddit').size().reset_index(name='clean_posts_count')
print("\nClean posts per subreddit:")
print(summary.to_string(index=False))

# Optional: Quick check to verify no [removed]/[deleted] remain
print("\nVerifying no [removed]/[deleted] posts remain:")
for sub in selected_subs[:3]:  # Check first 3 subreddits as a sample
    sub_df = combined_df[combined_df['subreddit'] == sub]
    sample_texts = sub_df['selftext'].head(3).tolist()
    print(f"\nr/{sub} sample selftexts:")
    for i, text in enumerate(sample_texts, 1):
        preview = text[:100] + "..." if len(text) > 100 else text
        print(f"  {i}. {preview}")

combined_df.head()


=== Processing r/technology (target: 15000 clean posts) ===
  Processed 100,000 posts, found 624 clean so far
  Processed 200,000 posts, found 1,637 clean so far
  Processed 300,000 posts, found 2,453 clean so far
  Processed 400,000 posts, found 2,935 clean so far
  Processed 500,000 posts, found 3,666 clean so far
  Processed 600,000 posts, found 4,167 clean so far
  Processed 700,000 posts, found 6,497 clean so far
  Processed 800,000 posts, found 6,822 clean so far
  Processed 900,000 posts, found 7,538 clean so far
  Processed 1,000,000 posts, found 8,241 clean so far
  Processed 1,100,000 posts, found 10,355 clean so far
  Processed 1,200,000 posts, found 10,583 clean so far
  Processed 1,300,000 posts, found 11,056 clean so far
  Processed 1,400,000 posts, found 11,547 clean so far
  Processed 1,500,000 posts, found 12,009 clean so far
  Processed 1,600,000 posts, found 12,951 clean so far
  Processed 1,700,000 posts, found 13,057 clean so far
  Processed 1,800,000 posts, found

,allow_live_comments,archived,author,author_fullname,banned_by,category,content_categories,contest_mode,created_utc,discussion_type,...,top_awarded_type,total_awards_received,treatment_tags,upvote_ratio,url,url_overridden_by_dest,view_count,whitelist_status,wls,subreddit
0,None,False,incognito1600,None,None,None,None,None,1454293865,None,...,None,None,None,None,https://www.reddit.com/r/technology/comments/4...,None,None,None,None,technology
1,None,False,mike_do,None,None,None,None,None,1454298603,None,...,None,None,None,None,https://www.reddit.com/r/technology/comments/4...,None,None,None,None,technology
2,None,False,Indyjones007,None,None,None,None,None,1454348428,None,...,None,None,None,None,https://www.reddit.com/r/technology/comments/4...,None,None,None,None,technology
3,None,False,arnoldswollenegger,None,None,None,None,None,1454368238,None,...,None,None,None,None,https://www.reddit.com/r/technology/comments/4...,None,None,None,None,technology
4,None,False,VideoGameAttorney,None,None,None,None,None,1454388312,None,...,None,None,None,None,https://www.reddit.com/r/technology/comments/4...,None,None,None,None,technology


In [ ]:
combined_df["subreddit"].value_counts()

,count
subreddit,
technology,15000
askscience,15000
todayilearned,15000
philosophy,15000
YouShouldKnow,15000
IWantToLearn,15000
personalfinance,15000
buildapc,15000
DIY,15000


In [ ]:
combined_df.to_csv("reddit_data2.csv", index=False)

In [ ]:
mask_removed_deleted = combined_df['selftext'].str.contains(r'\[removed\]|\[deleted\]',
                                                            case=False,
                                                            na=False,
                                                            regex=True)

# Add a column to identify these posts
combined_df['is_removed_deleted'] = mask_removed_deleted

# Calculate totals per subreddit
summary = combined_df.groupby('subreddit').agg(
    total_posts=('selftext', 'count'),
    removed_deleted_count=('is_removed_deleted', 'sum'),
    removed_deleted_percentage=('is_removed_deleted', lambda x: f"{(x.sum()/len(x)*100):.2f}%")
).reset_index()

# Display the summary
print("Summary of [removed]/[deleted] posts by subreddit:")
print("="*70)
print(summary.to_string(index=False))

Summary of [removed]/[deleted] posts by subreddit:
        subreddit  total_posts  removed_deleted_count removed_deleted_percentage
              DIY        15000                    960                      6.40%
     GetMotivated        15000                  11951                     79.67%
     IWantToLearn        15000                    267                      1.78%
      LifeProTips        15000                      1                      0.01%
   Showerthoughts        15000                      1                      0.01%
    YouShouldKnow        15000                   5293                     35.29%
       askscience        15000                    559                      3.73%
            books        15000                   2930                     19.53%
         buildapc        15000                    710                      4.73%
explainlikeimfive        15000                      0                      0.00%
          gadgets        15000                  11494     

In [ ]:
tier1_subs = [
    'askscience', 'books', 'buildapc', 'DIY', 'explainlikeimfive',
    'history', 'IWantToLearn', 'LifeProTips', 'personalfinance',
    'philosophy', 'Showerthoughts', 'tifu'
]
target_size = 10000
data_frames = []

for subreddit in tier1_subs:
    print(f"\n=== r/{subreddit} (target {target_size} clean posts) ===")
    stream = load_dataset(
        "HuggingFaceGECLM/REDDIT_submissions",
        split=subreddit,
        streaming=True,
        token=HF_TOKEN
    )

    clean_posts = []
    for idx, post in enumerate(stream):
        # Robust filter: non‑empty, not None, and no [removed]/[deleted]
        text = post.get('selftext', '')
        if text and isinstance(text, str) and text.strip():
            lower = text.strip().lower()
            if '[removed]' not in lower and '[deleted]' not in lower:
                clean_posts.append(post)
                if len(clean_posts) >= target_size:
                    print(f"  Reached target after processing {idx+1} posts.")
                    break

        # Optional progress every 100k posts
        if (idx + 1) % 100000 == 0:
            print(f"  Processed {idx+1} posts, found {len(clean_posts)} clean so far.")

    print(f"  Collected {len(clean_posts)} clean posts from r/{subreddit}")

    df_sub = pd.DataFrame(clean_posts)
    df_sub['subreddit'] = subreddit
    data_frames.append(df_sub)

# Combine all subreddits from this tier
combined_tier1 = pd.concat(data_frames, ignore_index=True)
print(f"\nTier 1 combined shape: {combined_tier1.shape}")
# Save or inspect
combined_tier1.head()


=== r/askscience (target 10000 clean posts) ===
  Reached target after processing 12544 posts.
  Collected 10000 clean posts from r/askscience

=== r/books (target 10000 clean posts) ===
  Reached target after processing 34346 posts.
  Collected 10000 clean posts from r/books

=== r/buildapc (target 10000 clean posts) ===
  Reached target after processing 11816 posts.
  Collected 10000 clean posts from r/buildapc

=== r/DIY (target 10000 clean posts) ===
  Reached target after processing 19827 posts.
  Collected 10000 clean posts from r/DIY

=== r/explainlikeimfive (target 10000 clean posts) ===
  Reached target after processing 15146 posts.
  Collected 10000 clean posts from r/explainlikeimfive

=== r/history (target 10000 clean posts) ===


KeyboardInterrupt: 

In [ ]:
tier2_subs = ['YouShouldKnow']   # only one from the summary
target_size = 5000
data_frames = []

for subreddit in tier2_subs:
    print(f"\n=== r/{subreddit} (target {target_size} clean posts) ===")
    stream = load_dataset(
        "HuggingFaceGECLM/REDDIT_submissions",
        split=subreddit,
        streaming=True,
        token=HF_TOKEN
    )

    clean_posts = []
    for idx, post in enumerate(stream):
        text = post.get('selftext', '')
        if text and isinstance(text, str) and text.strip():
            lower = text.strip().lower()
            if '[removed]' not in lower and '[deleted]' not in lower:
                clean_posts.append(post)
                if len(clean_posts) >= target_size:
                    print(f"  Reached target after processing {idx+1} posts.")
                    break

        if (idx + 1) % 100000 == 0:
            print(f"  Processed {idx+1} posts, found {len(clean_posts)} clean so far.")

    print(f"  Collected {len(clean_posts)} clean posts from r/{subreddit}")

    df_sub = pd.DataFrame(clean_posts)
    df_sub['subreddit'] = subreddit
    data_frames.append(df_sub)

combined_tier2 = pd.concat(data_frames, ignore_index=True)
print(f"\nTier 2 combined shape: {combined_tier2.shape}")
combined_tier2.head()

In [ ]:

# Subreddits in this tier
tier3_subs = ['GetMotivated', 'gadgets', 'space', 'todayilearned']
target_size = 3000
data_frames = []

for subreddit in tier3_subs:
    print(f"\n=== r/{subreddit} (target {target_size} clean posts) ===")
    stream = load_dataset(
        "HuggingFaceGECLM/REDDIT_submissions",
        split=subreddit,
        streaming=True,
        token=HF_TOKEN
    )

    clean_posts = []
    for idx, post in enumerate(stream):
        text = post.get('selftext', '')
        if text and isinstance(text, str) and text.strip():
            lower = text.strip().lower()
            if '[removed]' not in lower and '[deleted]' not in lower:
                clean_posts.append(post)
                if len(clean_posts) >= target_size:
                    print(f"  Reached target after processing {idx+1} posts.")
                    break

        if (idx + 1) % 100000 == 0:
            print(f"  Processed {idx+1} posts, found {len(clean_posts)} clean so far.")

    print(f"  Collected {len(clean_posts)} clean posts from r/{subreddit}")

    df_sub = pd.DataFrame(clean_posts)
    df_sub['subreddit'] = subreddit
    data_frames.append(df_sub)

combined_tier3 = pd.concat(data_frames, ignore_index=True)
print(f"\nTier 3 combined shape: {combined_tier3.shape}")
combined_tier3.head()

In [ ]:
["GetMotivated","YouShouldKnow","books","gadgets","history","philosophy"]